In [0]:
print('Ola Mundo')

Ola Mundo


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/dados/projeto_sst1/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/dados/projeto_sst1/acidentes_sst.csv,acidentes_sst.csv,1271,1791573895000
dbfs:/Volumes/workspace/default/dados/projeto_sst1/cat_emitidas.csv,cat_emitidas.csv,615,1791573895000
dbfs:/Volumes/workspace/default/dados/projeto_sst1/horas_trabalhadas.csv,horas_trabalhadas.csv,699,1791573894000
dbfs:/Volumes/workspace/default/dados/projeto_sst1/quase_acidentes.csv,quase_acidentes.csv,793,1791573895000


In [0]:
df_acidentes = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/dados/projeto_sst1/acidentes_sst.csv")
)

display(df_acidentes)

id,data,unidade,tipo_acidente,causa,dias_afastamento,gravidade
1,2026-01-05,Petrolina,Queda,Falta de atenção,3,Media
2,2026-01-12,Juazeiro,Corte,Ferramenta inadequada,0,Baixa
3,2026-01-18,Petrolina,Choque Eletrico,Falha de bloqueio,20,Alta
4,2026-02-02,Recife,Queda de Material,Falta de isolamento,5,Media
5,2026-02-15,Salvador,Corte,Ausencia de EPI,2,Media
6,2026-02-20,Petrolina,Torcao,Piso irregular,1,Baixa
7,2026-03-03,Juazeiro,Queda,Falta de atenção,4,Media
8,2026-03-15,Recife,Choque Eletrico,Falha operacional,15,Alta
9,2026-03-28,Salvador,Corte,Ferramenta inadequada,0,Baixa
10,2026-04-02,Petrolina,Queda,Falta de treinamento,7,Alta


In [0]:
df_horas = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/dados/projeto_sst1/horas_trabalhadas.csv")
)

display(df_horas)

mes,unidade,horas_trabalhadas
2026-01-01,Petrolina,28000
2026-01-01,Juazeiro,25000
2026-01-01,Recife,30000
2026-01-01,Salvador,32000
2026-02-01,Petrolina,29000
2026-02-01,Juazeiro,26000
2026-02-01,Recife,31000
2026-02-01,Salvador,33000
2026-03-01,Petrolina,29500
2026-03-01,Juazeiro,26500


In [0]:
df_quase = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/dados/projeto_sst1/quase_acidentes.csv")
)

display(df_quase)

id,data,unidade,descricao,classificacao
1,2026-01-10,Petrolina,Escada sem fixacao,Alto
2,2026-01-18,Juazeiro,Material armazenado inadequadamente,Medio
3,2026-02-05,Recife,Falta de sinalizacao,Alto
4,2026-02-17,Salvador,Cabo exposto,Alto
5,2026-03-09,Petrolina,Veiculo sem cone de isolamento,Medio
6,2026-03-23,Juazeiro,Ferramenta danificada,Baixo
7,2026-04-08,Recife,Piso escorregadio,Alto
8,2026-04-19,Salvador,EPI incorreto,Medio
9,2026-05-10,Petrolina,Falta de APR,Alto
10,2026-05-28,Juazeiro,Área sem bloqueio,Alto


In [0]:
df_cat = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/dados/projeto_sst1/cat_emitidas.csv")
)

display(df_cat)

id,data,unidade,colaborador,tipo_acidente,cat_emitida
1,2026-01-05,Petrolina,Funcionario 001,Queda,Sim
2,2026-01-18,Petrolina,Funcionario 002,Choque Eletrico,Sim
3,2026-02-15,Salvador,Funcionario 003,Corte,Sim
4,2026-03-15,Recife,Funcionario 004,Choque Eletrico,Sim
5,2026-04-02,Petrolina,Funcionario 005,Queda,Sim
6,2026-04-22,Salvador,Funcionario 006,Queda de Material,Sim
7,2026-05-12,Petrolina,Funcionario 007,Choque Eletrico,Sim
8,2026-05-24,Juazeiro,Funcionario 008,Queda,Sim
9,2026-07-03,Juazeiro,Funcionario 009,Queda de Material,Sim
10,2026-07-22,Salvador,Funcionario 010,Choque Eletrico,Sim


In [0]:
df_acidentes.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_acidentes")

df_horas.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_horas")

df_quase.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_quase_acidentes")

df_cat.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_cat_emitidas")


In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,bronze_acidentes,false
default,bronze_cat_emitidas,false
default,bronze_horas,false
default,bronze_quase_acidentes,false
default,clientes,false


In [0]:
from pyspark.sql.functions import *

# Acidentes
silver_acidentes = (
    spark.table("bronze_acidentes")
    .dropDuplicates()
)

silver_acidentes.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_acidentes")

# Horas
silver_horas = (
    spark.table("bronze_horas")
    .dropDuplicates()
)

silver_horas.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_horas")

# CAT
silver_cat = (
    spark.table("bronze_cat_emitidas")
    .dropDuplicates()
)

silver_cat.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_cat_emitidas")

# Quase acidentes
silver_quase = (
    spark.table("bronze_quase_acidentes")
    .dropDuplicates()
)

silver_quase.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_quase_acidentes")

In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,bronze_acidentes,false
default,bronze_cat_emitidas,false
default,bronze_horas,false
default,bronze_quase_acidentes,false
default,clientes,false
default,silver_acidentes,false
default,silver_cat_emitidas,false
default,silver_horas,false
default,silver_quase_acidentes,false


In [0]:
from pyspark.sql.functions import count

gold_acidentes_unidade = (
    spark.table("silver_acidentes")
    .groupBy("unidade")
    .agg(
        count("*").alias("qtd_acidentes")
    )
)

gold_acidentes_unidade.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_acidentes_unidade")

In [0]:
gold_gravidade = (
    spark.table("silver_acidentes")
    .groupBy("gravidade")
    .agg(
        count("*").alias("total")
    )
)

gold_gravidade.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_acidentes_gravidade")

In [0]:
gold_causas = (
    spark.table("silver_acidentes")
    .groupBy("causa")
    .agg(
        count("*").alias("ocorrencias")
    )
)

gold_causas.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_causas")

In [0]:
%sql
SELECT * FROM gold_acidentes_unidade;

unidade,qtd_acidentes
Recife,4
Petrolina,6
Juazeiro,5
Salvador,5


In [0]:
%sql
SELECT * FROM gold_acidentes_gravidade;

gravidade,total
Media,6
Baixa,7
Alta,7


In [0]:
%sql
SELECT * FROM gold_causas;


causa,ocorrencias
Piso irregular,2
Ausencia de EPI,2
Falta de treinamento,2
Esforço excessivo,1
Ferramenta inadequada,3
Falta de isolamento,3
Falta de atenção,3
Falha operacional,2
Falha de bloqueio,2


In [0]:
%sql
CREATE OR REPLACE TABLE gold_taxa_frequencia AS
SELECT
    a.unidade,
    COUNT(*) AS acidentes,
    SUM(h.horas_trabalhadas) AS horas,
    (COUNT(*) * 1000000.0) / SUM(h.horas_trabalhadas) AS taxa_frequencia
FROM silver_acidentes a
JOIN silver_horas h
    ON a.unidade = h.unidade
GROUP BY a.unidade

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_taxa_gravidade AS
SELECT
    a.unidade,
    SUM(a.dias_afastamento) AS dias_perdidos,
    SUM(h.horas_trabalhadas) AS horas,
    (SUM(a.dias_afastamento) * 1000000.0) /
    SUM(h.horas_trabalhadas) AS taxa_gravidade
FROM silver_acidentes a
JOIN silver_horas h
    ON a.unidade = h.unidade
GROUP BY a.unidade

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * 
FROM gold_taxa_frequencia;

unidade,acidentes,horas,taxa_frequencia
Recife,28,894000,31.3199105145
Petrolina,42,1257000,33.4128878282
Juazeiro,35,942500,37.1352785146
Salvador,35,1187500,29.4736842105


In [0]:
%sql
SELECT DISTINCT unidade
FROM silver_acidentes;

unidade
Petrolina
Salvador
Juazeiro
Recife


In [0]:
%sql
SELECT DISTINCT unidade
FROM silver_horas;

unidade
Recife
Petrolina
Juazeiro
Salvador


In [0]:
%sql
SELECT
    causa,
    COUNT(*) AS qtd_ocorrencias
FROM silver_acidentes
GROUP BY causa
ORDER BY qtd_ocorrencias DESC

causa,qtd_ocorrencias
Falta de isolamento,3
Falta de atenção,3
Ferramenta inadequada,3
Falha de bloqueio,2
Falta de treinamento,2
Piso irregular,2
Falha operacional,2
Ausencia de EPI,2
Esforço excessivo,1


In [0]:
%sql
SELECT
    gravidade,
    COUNT(*) AS total
FROM silver_acidentes
GROUP BY gravidade
ORDER BY total DESC

gravidade,total
Baixa,7
Alta,7
Media,6


In [0]:
%sql
SELECT
    gravidade,
    COUNT(*) AS total
FROM silver_acidentes
GROUP BY gravidade
ORDER BY total DESC

gravidade,total
Baixa,7
Alta,7
Media,6


In [0]:
%sql
SELECT
    unidade,
    SUM(dias_afastamento) AS dias_perdidos
FROM silver_acidentes
GROUP BY unidade
ORDER BY dias_perdidos DESC

unidade,dias_perdidos
Petrolina,49
Recife,26
Salvador,23
Juazeiro,22


In [0]:
%sql
CREATE OR REPLACE TABLE gold_taxa_frequencia AS

SELECT
    a.unidade,
    COUNT(*) AS acidentes,
    SUM(h.horas_trabalhadas) AS horas,
    ROUND(
        (COUNT(*) * 1000000.0) /
        SUM(h.horas_trabalhadas),
        2
    ) AS taxa_frequencia
FROM silver_acidentes a
JOIN silver_horas h
ON a.unidade = h.unidade
GROUP BY a.unidade;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_taxa_gravidade AS

SELECT
    a.unidade,
    SUM(a.dias_afastamento) AS dias_perdidos,
    SUM(h.horas_trabalhadas) AS horas,
    ROUND(
        (SUM(a.dias_afastamento) * 1000000.0) /
        SUM(h.horas_trabalhadas),
        2
    ) AS taxa_gravidade
FROM silver_acidentes a
JOIN silver_horas h
ON a.unidade = h.unidade
GROUP BY a.unidade;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,bronze_acidentes,false
default,bronze_cat_emitidas,false
default,bronze_horas,false
default,bronze_quase_acidentes,false
default,clientes,false
default,gold_acidentes_gravidade,false
default,gold_acidentes_unidade,false
default,gold_causas,false
default,gold_taxa_frequencia,false
default,gold_taxa_gravidade,false


In [0]:
%sql
SELECT *
FROM silver_acidentes;

id,data,unidade,tipo_acidente,causa,dias_afastamento,gravidade
6,2026-02-20,Petrolina,Torcao,Piso irregular,1,Baixa
9,2026-03-28,Salvador,Corte,Ferramenta inadequada,0,Baixa
5,2026-02-15,Salvador,Corte,Ausencia de EPI,2,Media
7,2026-03-03,Juazeiro,Queda,Falta de atenção,4,Media
14,2026-05-12,Petrolina,Choque Eletrico,Falha de bloqueio,18,Alta
2,2026-01-12,Juazeiro,Corte,Ferramenta inadequada,0,Baixa
3,2026-01-18,Petrolina,Choque Eletrico,Falha de bloqueio,20,Alta
12,2026-04-22,Salvador,Queda de Material,Falta de isolamento,8,Alta
13,2026-05-01,Recife,Corte,Ausencia de EPI,1,Baixa
10,2026-04-02,Petrolina,Queda,Falta de treinamento,7,Alta


In [0]:
%sql
SELECT *
FROM gold_taxa_frequencia;

unidade,acidentes,horas,taxa_frequencia
Recife,28,894000,31.32
Petrolina,42,1257000,33.41
Juazeiro,35,942500,37.14
Salvador,35,1187500,29.47


In [0]:
%sql
SELECT *
FROM gold_taxa_gravidade;

unidade,dias_perdidos,horas,taxa_gravidade
Recife,182,894000,203.58
Petrolina,343,1257000,272.87
Juazeiro,154,942500,163.40
Salvador,161,1187500,135.58


In [0]:
%sql
SELECT *
FROM gold_causas;

causa,ocorrencias
Piso irregular,2
Ausencia de EPI,2
Falta de treinamento,2
Esforço excessivo,1
Ferramenta inadequada,3
Falta de isolamento,3
Falta de atenção,3
Falha operacional,2
Falha de bloqueio,2


In [0]:
%sql
SELECT *
FROM gold_acidentes_unidade;

unidade,qtd_acidentes
Recife,4
Petrolina,6
Juazeiro,5
Salvador,5
